# RAG pipeline з SQLite

## Ноутбук для запитів

### Підключення до бази даних

Підключаємось до бази даних.

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

Перевіримо кількість документів в індекі.

In [2]:
sqlite_index.count()

188

Спробуємо, як працює пошук.

In [3]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'I just discovered the course. Can I still join?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?']

### RAG з sqlitesearch

Оскільки наш RAG модульний, ми просто замінимо індекс пошуку. Решта коду залишається незмінною.

In [5]:
from rag_helper import RAGBase
from google import genai
from dotenv import load_dotenv

load_dotenv()
gemini_client = genai.Client()

assistant = RAGBase(sqlite_index, gemini_client)

Задамо питання.

In [6]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

/workspaces/LLM_Zoomcamp_2026/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


Коли закінчимо роботу, закриваємо з'єднання з базою даних. Або можна почекати, коли Python сам виконає очистку після того, як ядро ноутбука вимкнеться.

In [3]:
sqlite_index.close()